In [1]:
%load_ext autoreload
%autoreload 2

# Notebook 12: NF template convergence for the DC4 Crab binning (component 01)

Finds integration settings of `NFBinnedTemplate` for the DC4 continuum binning (`dataio_continuum.yaml`:
10 Em bins from 100 keV to 10 MeV, Phi 6°, NSIDE 8), as notebook 09 did for the 511 keV binning.

**Criterion:** a setting is converged when a finer one changes the expected counts by less than a tenth of
the Poisson noise, |N_finer − N_current| / √N < 0.1 in every bin (or at least 99%), with N the expected counts in
the DC4 time window. Then the integration error shifts χ²/dof by less than 0.01.

Settings that pass go to `config/nf_template_settings.yaml` (case `dc4_crab_continuum`).

## Imports

In [2]:
import torch                                   # first: imported after histpy / cosipy, building the NF model crashes
import sys
from pathlib import Path
import numpy as np
import yaml                                    # reads the template settings file
import healpy as hp                            # pixel positions
import astropy.units as u
from histpy import Axes, Axis, HealpixAxis     # the binning given to the class
from cosipy.data_io import BinnedData           # reads the DC4 binning yaml (edges, time window)
from cosipy.background_estimation.ml.NFBkgNormalizationMap import NFBkgNormalizationMap   # Pascal's background lines
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent.parent))     # project root, so `src` is importable
from src.nf_template import NFBinnedTemplate   # the template class
from src.plot_style import COLORS, set_global_style
set_global_style()                             # after the cosipy import

torch.set_num_threads(4)                       # fastest on the MacBook (measured earlier)

%matplotlib inline


WARNING FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.



## Configuration

In [3]:
# Template settings file: the block for this binning (status, binning; quadrature still to be found)
SETTINGS_PATH = Path("../../config/nf_template_settings.yaml")
CASE = "dc4_crab_continuum"
with open(SETTINGS_PATH) as f:
    settings = yaml.safe_load(f)[CASE]
print(f"{CASE}: {settings['status']}")

# NF model: component 01, version 3.00; orientation file (attitude + live time)
MODEL_PATH = Path("../../model/nfbackground_part-01_v3-00.fits")
ORI_PATH = Path("../../data/DC4_orientation.ori")

# DC4 continuum binning yaml: time window, and a check that the settings file has the same binning
BINNING_YAML = Path("/Users/maura/Documents/COSI/data-challenge-4/workflow/data/config/dataio_continuum.yaml")
cfg = BinnedData(input_yaml=str(BINNING_YAML))
TMIN, TMAX = cfg.tmin, cfg.tmax

binning = settings["binning"]
print("Em edges match dataio_continuum.yaml:", np.allclose(binning["em_edges_kev"], cfg.energy_bins))
print("Phi bin, NSIDE, scheme match:", (binning["phi_bin_deg"], binning["nside"], binning["scheme"])
      == (cfg.phi_pix_size, cfg.nside, cfg.scheme.lower()))

EM_EDGES_KEV = np.array(binning["em_edges_kev"])                 # 10 energy bins, 100 keV - 10 MeV
PHI_EDGES_DEG = np.arange(0, 181, binning["phi_bin_deg"])        # 30 bins of 6 deg
NSIDE, SCHEME = binning["nside"], binning["scheme"]              # PsiChi: 768 pixels

# Convergence criterion: |N_finer - N_current| / sqrt(N) < CRITERION
CRITERION = 0.1

print(f"time window: {TMIN} to {TMAX} s ({(TMAX - TMIN) / 86400:.1f} days)")

dc4_crab_continuum: not validated
Em edges match dataio_continuum.yaml: True
Phi bin, NSIDE, scheme match: True
time window: 1835487300.036206 to 1843467254.887687 s (92.4 days)


## Binning

In [4]:
# Binning given to the class: three axes with the labels it expects ("Em", "Phi", "PsiChi")
axes = Axes([Axis(EM_EDGES_KEV * u.keV, label="Em"),
             Axis(PHI_EDGES_DEG * u.deg, label="Phi"),
             HealpixAxis(nside=NSIDE, scheme=SCHEME, label="PsiChi")])

for ax in axes:
    print(f"{ax.label:6s}: {ax.nbins} bins")

Em    : 10 bins
Phi   : 30 bins
PsiChi: 768 bins


***
## 1. Em convergence

More Em points per bin must change the counts less and less. Only Em changes: Phi uses the FINAL settings of
the 511 keV case, the sky a cheap level (k = 1), on a few ordinary test pixels (away from the poles and from χ = 0).
The 511 keV line falls inside the 398–631 keV bin, so that bin is split at 511 keV (`em_breaks_kev`).

First the cost of each setting, before computing anything.

In [5]:
# Template object: loads the NF model (default pole / seam settings)
template = NFBinnedTemplate(MODEL_PATH)

# Test pixels: ordinary pixels spread in psi (30-150 deg) and chi (45, 135 deg), RING numbering
TEST_PIXELS = np.array([hp.ang2pix(NSIDE, np.deg2rad(psi), np.deg2rad(chi))
                        for psi in (30, 60, 90, 120, 150) for chi in (45, 135)])

# Fixed during this test: sky level k = 1, Phi as in the 511 keV case (FINAL)
K_TEST = 1
PHI_QUAD = dict(n_phi_per_bin=4, phi_breaks_deg=[15.5, 16.0, 16.5, 17.0])

# Em settings to compare: more points per bin each time
EM_POINTS = [4, 8, 16]

# Cost of each setting (density evaluations; nothing is computed)
SPEED = 7000                                                     # evaluations/s on the MacBook
for n_em in EM_POINTS:
    n = template.n_evaluations(axes, TEST_PIXELS, subpixel_level=K_TEST,
                               n_em_per_bin=n_em, em_breaks_kev=[511.0], **PHI_QUAD)
    print(f"n_em_per_bin = {n_em:2d}: {n:>10,} evaluations, ~{n / SPEED / 60:4.1f} min")

n_em_per_bin =  4:    239,360 evaluations, ~ 0.6 min
n_em_per_bin =  8:    478,720 evaluations, ~ 1.1 min
n_em_per_bin = 16:    957,440 evaluations, ~ 2.3 min


In [6]:
# Expected counts N = R * T * P on the test pixels, for each Em setting (~5-7 min in total).
# Each new setting changes the binning, so the class discards the previous results (it logs a warning: expected).
N_em = {}
for n_em in EM_POINTS:
    counts = template.get_binned_background(axes, "local", ORI_PATH, TMIN, TMAX, pixels=TEST_PIXELS,
                                            subpixel_level=K_TEST, n_em_per_bin=n_em, em_breaks_kev=[511.0],
                                            **PHI_QUAD)
    N_em[n_em] = counts.contents[:, :, TEST_PIXELS]               # counts, (Em bins, Phi bins, test pixels)


WARNING UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/BucketizationUtils.h:34.)

Binning or settings changed: results of 10 computed pixel(s) discarded.
Binning or settings changed: results of 10 computed pixel(s) discarded.


In [7]:
# Integration error of each setting, in units of the Poisson noise of the bin:
#     z = |N(n points) - N(16 points)| / sqrt(N(16 points))
# The 16-point counts are the reference. Converged when z < CRITERION (0.1) in every bin.
# Only bins with counts (Phi below ~10 deg is empty).
N_ref = N_em[EM_POINTS[-1]]
has_counts = N_ref > 0
n_bins = has_counts.sum()

print(f"All {n_bins} bins (10 Em x 30 Phi x 10 test pixels, with counts):")
for n_em in EM_POINTS[:-1]:
    z = np.abs(N_em[n_em] - N_ref)[has_counts] / np.sqrt(N_ref[has_counts])
    print(f"  {n_em:2d} points per Em bin: worst bin z = {z.max():6.3f} | 99% of bins below z = {np.percentile(z, 99):6.3f} "
          f"| bins that pass (z < {CRITERION}): {np.mean(z < CRITERION):5.1%}")

# The same per energy bin: the worst z among its Phi bins and test pixels
print(f"\nWorst z in each energy bin (criterion: < {CRITERION}):")
print(f"  {'Em bin [keV]':>17s}   " + "   ".join(f"{n:2d} points" for n in EM_POINTS[:-1]))
for a, (lo, hi) in enumerate(zip(EM_EDGES_KEV[:-1], EM_EDGES_KEV[1:])):
    ok = has_counts[a]
    worst = [np.max(np.abs(N_em[n][a] - N_ref[a])[ok] / np.sqrt(N_ref[a][ok])) for n in EM_POINTS[:-1]]
    print(f"  {lo:7.1f} - {hi:7.1f}   " + "   ".join(f"{v:9.3f}" for v in worst))

All 3000 bins (10 Em x 30 Phi x 10 test pixels, with counts):
   4 points per Em bin: worst bin z = 18.907 | 99% of bins below z =  5.643 | bins that pass (z < 0.1): 60.7%
   8 points per Em bin: worst bin z =  4.363 | 99% of bins below z =  1.690 | bins that pass (z < 0.1): 76.5%

Worst z in each energy bin (criterion: < 0.1):
       Em bin [keV]    4 points    8 points
    100.0 -   158.5      18.907       2.969
    158.5 -   251.2       3.498       2.033
    251.2 -   398.1      11.387       4.363
    398.1 -   631.0       0.943       0.362
    631.0 -  1000.0       0.560       0.227
   1000.0 -  1584.9       0.487       0.189
   1584.9 -  2511.9       1.726       0.657
   2511.9 -  3981.1       0.513       0.197
   3981.1 -  6309.6       0.259       0.114
   6309.6 - 10000.0       0.079       0.025


### 1.2 Em points around Pascal's background lines

Pascal's `NFBkgNormalizationMap` (his branch) lists 37 background lines, each with a half-width
W = √E × 0.30 or × 0.40 keV. Each Em bin is split at E − W, E and E + W of every line inside it (`em_breaks_kev`),
and each piece gets its own Gauss-Legendre points. Same test pixels, Phi and sky level as in 1.1.

First the line list and the cost.

In [8]:
# Pascal's default background lines and widths (no NF density needed, only his settings)
line_map = NFBkgNormalizationMap()
LINES_KEV = np.array([peak.energy_keV for peak in line_map.Em_peaks])                          # line centres E [keV]
HALF_WIDTHS_KEV = np.array([np.sqrt(peak.energy_keV) * line_map.Em_peak_widths[peak.width_type]
                            for peak in line_map.Em_peaks])                                    # W = sqrt(E) * 0.30 or 0.40
print(f"{len(LINES_KEV)} lines, {LINES_KEV.min():.0f}-{LINES_KEV.max():.0f} keV; half-widths {HALF_WIDTHS_KEV.min():.1f}-{HALF_WIDTHS_KEV.max():.1f} keV")

# Break points: E - W, E, E + W of every line, inside the binning range
LINE_BREAKS_KEV = np.unique(np.concatenate([LINES_KEV - HALF_WIDTHS_KEV, LINES_KEV, LINES_KEV + HALF_WIDTHS_KEV]))
LINE_BREAKS_KEV = LINE_BREAKS_KEV[(LINE_BREAKS_KEV > EM_EDGES_KEV[0]) & (LINE_BREAKS_KEV < EM_EDGES_KEV[-1])]

# Lines in each Em bin
for lo, hi in zip(EM_EDGES_KEV[:-1], EM_EDGES_KEV[1:]):
    inside = LINES_KEV[(LINES_KEV > lo) & (LINES_KEV < hi)]
    print(f"  {lo:7.1f} - {hi:7.1f} keV: {len(inside):2d} lines {np.round(inside, 1).tolist()}")

# Points per piece to compare; the finest is the reference
LINE_POINTS = [2, 4, 8]

# Cost of each setting (density evaluations; nothing is computed)
for n_em in LINE_POINTS:
    n = template.n_evaluations(axes, TEST_PIXELS, subpixel_level=K_TEST,
                               n_em_per_bin=n_em, em_breaks_kev=LINE_BREAKS_KEV, **PHI_QUAD)
    print(f"{n_em} points per piece: {n:>11,} evaluations, ~{n / SPEED / 60:4.1f} min")

37 lines, 144-2822 keV; half-widths 3.6-15.9 keV
    100.0 -   158.5 keV:  1 lines [144.0]
    158.5 -   251.2 keV:  3 lines [175.0, 184.5, 198.0]
    251.2 -   398.1 keV:  3 lines [300.0, 310.0, 397.5]
    398.1 -   631.0 keV:  7 lines [427.0, 438.0, 472.0, 511.0, 574.0, 584.0, 595.5]
    631.0 -  1000.0 keV:  5 lines [752.0, 810.0, 843.0, 872.0, 987.0]
   1000.0 -  1584.9 keV:  7 lines [1014.0, 1039.0, 1076.0, 1108.0, 1335.0, 1367.0, 1432.0]
   1584.9 -  2511.9 keV:  9 lines [1610.0, 1632.0, 1763.0, 1778.0, 1809.0, 1893.0, 1921.0, 2208.0, 2240.0]
   2511.9 -  3981.1 keV:  2 lines [2750.0, 2822.0]
   3981.1 -  6309.6 keV:  0 lines []
   6309.6 - 10000.0 keV:  0 lines []
2 points per piece:   1,316,480 evaluations, ~ 3.1 min
4 points per piece:   2,632,960 evaluations, ~ 6.3 min
8 points per piece:   5,265,920 evaluations, ~12.5 min


In [9]:
# Expected counts N = R * T * P on the test pixels, with points around the lines (~22 min in total)
N_lines = {}
for n_em in LINE_POINTS:
    counts = template.get_binned_background(axes, "local", ORI_PATH, TMIN, TMAX, pixels=TEST_PIXELS,
                                            subpixel_level=K_TEST, n_em_per_bin=n_em, em_breaks_kev=LINE_BREAKS_KEV,
                                            **PHI_QUAD)
    N_lines[n_em] = counts.contents[:, :, TEST_PIXELS]            # counts, (Em bins, Phi bins, test pixels)

Binning or settings changed: results of 10 computed pixel(s) discarded.
Binning or settings changed: results of 10 computed pixel(s) discarded.
Binning or settings changed: results of 10 computed pixel(s) discarded.


In [10]:
# Integration error of each setting, in units of the Poisson noise of the bin:
#     z = |N(n points per piece) - N(8 points per piece)| / sqrt(N(8 points per piece))
# The 8-point counts are the reference. Converged when z < CRITERION (0.1) in every bin.
N_ref_lines = N_lines[LINE_POINTS[-1]]
has_counts = N_ref_lines > 0

print(f"All {has_counts.sum()} bins (10 Em x 30 Phi x 10 test pixels, with counts):")
for n_em in LINE_POINTS[:-1]:
    z = np.abs(N_lines[n_em] - N_ref_lines)[has_counts] / np.sqrt(N_ref_lines[has_counts])
    print(f"  {n_em} points per piece: worst bin z = {z.max():6.3f} | 99% of bins below z = {np.percentile(z, 99):6.3f} "
          f"| bins that pass (z < {CRITERION}): {np.mean(z < CRITERION):5.1%}")

# The same per energy bin: the worst z among its Phi bins and test pixels
print(f"\nWorst z in each energy bin (criterion: < {CRITERION}):")
print(f"  {'Em bin [keV]':>17s}   " + "   ".join(f"{n} per piece" for n in LINE_POINTS[:-1]))
for a, (lo, hi) in enumerate(zip(EM_EDGES_KEV[:-1], EM_EDGES_KEV[1:])):
    ok = has_counts[a]
    worst = [np.max(np.abs(N_lines[n][a] - N_ref_lines[a])[ok] / np.sqrt(N_ref_lines[a][ok])) for n in LINE_POINTS[:-1]]
    print(f"  {lo:7.1f} - {hi:7.1f}   " + "   ".join(f"{v:11.3f}" for v in worst))

All 3000 bins (10 Em x 30 Phi x 10 test pixels, with counts):
  2 points per piece: worst bin z = 29.216 | 99% of bins below z =  4.594 | bins that pass (z < 0.1): 74.5%
  4 points per piece: worst bin z = 11.465 | 99% of bins below z =  2.107 | bins that pass (z < 0.1): 81.2%

Worst z in each energy bin (criterion: < 0.1):
       Em bin [keV]   2 per piece   4 per piece
    100.0 -   158.5        29.216        11.465
    158.5 -   251.2         2.108         2.219
    251.2 -   398.1        12.138         3.120
    398.1 -   631.0         0.249         0.093
    631.0 -  1000.0         0.247         0.196
   1000.0 -  1584.9         0.112         0.066
   1584.9 -  2511.9         1.271         0.273
   2511.9 -  3981.1         0.558         1.046
   3981.1 -  6309.6         1.345         0.233
   6309.6 - 10000.0         0.175         0.095
